<a href="https://colab.research.google.com/github/shaikfarzana12/231FA04G87-MLOps-Feast-SkillGap/blob/main/Student_SkillGap_Feast_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLA 1 — Curriculum-Industry Skill-Gap Feature Store Using Feast

**Dataset:** `cse_skill_dataset (4)(1).csv`  
**Objective:** Build a simple Feast feature store from the student skill-gap dataset, retrieve historical features, materialize them into an online store, retrieve online features, and use them in a machine-learning model.



## Project workflow

```text
Student Skill Dataset
        ↓
Data checking + preprocessing
        ↓
Feature engineering
        ↓
Parquet offline data
        ↓
Feast Entity + Data Source + FeatureView
        ↓
feast apply
        ↓
Historical feature retrieval
        ↓
ML model training + accuracy
        ↓
Materialization
        ↓
SQLite online store
        ↓
Online feature retrieval
        ↓
Final prediction
```

In [1]:
# STEP 1 — Install the required libraries
!pip install -q feast pyarrow scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.


In [2]:
# STEP 2 — Import libraries
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# STEP 3 — Upload the student CSV file
from google.colab import files

uploaded = files.upload()
csv_name = next(iter(uploaded))

print("Uploaded file:", csv_name)

Saving cse_skill_dataset (4).csv to cse_skill_dataset (4).csv
Uploaded file: cse_skill_dataset (4).csv


In [4]:
# STEP 4 — Load the dataset
df = pd.read_csv(csv_name)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Rows: 150
Columns: 15


,Student_ID,Specialization,CGPA,Programming_Skill_Score,Communication_Skill_Score,Certifications_Count,Internship_Experience,Projects_Completed,DSA_Platform_Rating,Placement_Training_Attended,Soft_Skills_Score,Curriculum_Coverage_Percent,Industry_Demand_Score,Curriculum_Industry_Alignment_Score,Skill_Gap_Label
0,CSE2027001,Cloud Computing,8.25,33,65,1,Yes,2,1009,Yes,84,42,51,94.26,Yes
1,CSE2027002,Core CSE,6.50,33,55,5,No,3,1719,No,50,84,77,94.85,Yes
2,CSE2027003,Cloud Computing,6.17,73,43,0,No,1,1535,No,63,91,52,65.22,Yes
3,CSE2027004,IoT,7.81,78,40,4,No,5,1982,Yes,38,42,92,65.44,No
4,CSE2027005,IoT,6.74,40,59,6,Yes,6,1369,No,76,50,73,82.93,Yes


In [5]:
# STEP 5 — Understand the dataset
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Shape: (150, 15)

Column names:
['Student_ID', 'Specialization', 'CGPA', 'Programming_Skill_Score', 'Communication_Skill_Score', 'Certifications_Count', 'Internship_Experience', 'Projects_Completed', 'DSA_Platform_Rating', 'Placement_Training_Attended', 'Soft_Skills_Score', 'Curriculum_Coverage_Percent', 'Industry_Demand_Score', 'Curriculum_Industry_Alignment_Score', 'Skill_Gap_Label']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Student_ID                           150 non-null    object 
 1   Specialization                       150 non-null    object 
 2   CGPA                                 150 non-null    float64
 3   Programming_Skill_Score              150 non-null    int64  
 4   Communication_Skill_Score            150 non-null    int64  
 5   Certifications_Co

In [6]:
# STEP 6 — Statistical summary
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Student_ID,150,150,CSE2027001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Specialization,150,6,Cloud Computing,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CGPA,150.0,NaN,NaN,NaN,7.5306,1.234594,5.51,6.535,7.375,8.6175,9.78
Programming_Skill_Score,150.0,NaN,NaN,NaN,64.453333,21.05875,30.0,45.25,65.5,82.0,100.0
Communication_Skill_Score,150.0,NaN,NaN,NaN,65.76,19.241552,30.0,49.25,65.0,82.75,99.0
Certifications_Count,150.0,NaN,NaN,NaN,2.96,2.065634,0.0,1.0,3.0,5.0,6.0
Internship_Experience,150,2,No,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Projects_Completed,150.0,NaN,NaN,NaN,3.88,2.543369,0.0,2.0,4.0,6.0,8.0
DSA_Platform_Rating,150.0,NaN,NaN,NaN,1507.78,380.417801,816.0,1202.5,1516.0,1832.5,2188.0
Placement_Training_Attended,150,2,No,84,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# STEP 7 — Check missing values
missing = df.isnull().sum()

print("Missing values in each column:")
display(missing.to_frame("Missing Values"))

if missing.sum() == 0:
    print("Good news: this dataset has no missing values.")
else:
    print("Missing values were found and will be handled in preprocessing.")

Missing values in each column:


,Missing Values
Student_ID,0
Specialization,0
CGPA,0
Programming_Skill_Score,0
Communication_Skill_Score,0
Certifications_Count,0
Internship_Experience,0
Projects_Completed,0
DSA_Platform_Rating,0
Placement_Training_Attended,0


Good news: this dataset has no missing values.


In [8]:
# STEP 8 — Check the target variable
print("Target column: Skill_Gap_Label")
display(df["Skill_Gap_Label"].value_counts())

print("\nTarget percentages:")
display(df["Skill_Gap_Label"].value_counts(normalize=True).mul(100).round(2))

Target column: Skill_Gap_Label


,count
Skill_Gap_Label,
Yes,75
No,75



Target percentages:


,proportion
Skill_Gap_Label,
Yes,50.0
No,50.0


In [9]:
# STEP 9 — Check categorical columns
categorical_check = ["Specialization", "Internship_Experience",
                     "Placement_Training_Attended", "Skill_Gap_Label"]

for col in categorical_check:
    print(f"\n{col}:")
    print(df[col].value_counts())


Specialization:
Specialization
Cloud Computing    32
Core CSE           32
IoT                24
Data Science       22
AI & ML            22
Cyber Security     18
Name: count, dtype: int64

Internship_Experience:
Internship_Experience
No     82
Yes    68
Name: count, dtype: int64

Placement_Training_Attended:
Placement_Training_Attended
No     84
Yes    66
Name: count, dtype: int64

Skill_Gap_Label:
Skill_Gap_Label
Yes    75
No     75
Name: count, dtype: int64


In [10]:
# STEP 10 — Create engineered features
feature_df = df.copy()

# Robust missing-value handling.
numeric_columns = [
    "CGPA",
    "Programming_Skill_Score",
    "Communication_Skill_Score",
    "Certifications_Count",
    "Projects_Completed",
    "DSA_Platform_Rating",
    "Soft_Skills_Score",
    "Curriculum_Coverage_Percent",
    "Industry_Demand_Score",
    "Curriculum_Industry_Alignment_Score"
]

categorical_columns = [
    "Specialization",
    "Internship_Experience",
    "Placement_Training_Attended"
]

for col in numeric_columns:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

for col in categorical_columns:
    feature_df[col] = feature_df[col].fillna(feature_df[col].mode()[0])

# Encode Yes/No columns
feature_df["Internship_Flag"] = (
    feature_df["Internship_Experience"].map({"Yes": 1, "No": 0}).astype("int64")
)

feature_df["Placement_Training_Flag"] = (
    feature_df["Placement_Training_Attended"].map({"Yes": 1, "No": 0}).astype("int64")
)

# Create a simple experience score from 0 to 100
feature_df["Experience_Score"] = (
    feature_df["Internship_Flag"] * 40
    + feature_df["Placement_Training_Flag"] * 30
    + (feature_df["Projects_Completed"] / 8) * 20
    + (feature_df["Certifications_Count"] / 6) * 10
).astype("float32")

# Rename the entity column
feature_df = feature_df.rename(columns={"Student_ID": "student_id"})

display(feature_df.head())

,student_id,Specialization,CGPA,Programming_Skill_Score,Communication_Skill_Score,Certifications_Count,Internship_Experience,Projects_Completed,DSA_Platform_Rating,Placement_Training_Attended,Soft_Skills_Score,Curriculum_Coverage_Percent,Industry_Demand_Score,Curriculum_Industry_Alignment_Score,Skill_Gap_Label,Internship_Flag,Placement_Training_Flag,Experience_Score
0,CSE2027001,Cloud Computing,8.25,33,65,1,Yes,2,1009,Yes,84,42,51,94.26,Yes,1,1,76.666664
1,CSE2027002,Core CSE,6.50,33,55,5,No,3,1719,No,50,84,77,94.85,Yes,0,0,15.833333
2,CSE2027003,Cloud Computing,6.17,73,43,0,No,1,1535,No,63,91,52,65.22,Yes,0,0,2.500000
3,CSE2027004,IoT,7.81,78,40,4,No,5,1982,Yes,38,42,92,65.44,No,0,1,49.166668
4,CSE2027005,IoT,6.74,40,59,6,Yes,6,1369,No,76,50,73,82.93,Yes,1,0,65.000000


In [11]:
# STEP 11 — Add synthetic timestamps for Feast
feature_df["event_timestamp"] = pd.date_range(
    start="2025-01-01",
    periods=len(feature_df),
    freq="D",
    tz="UTC"
)

display(feature_df[["student_id", "event_timestamp"]].head())

,student_id,event_timestamp
0,CSE2027001,2025-01-01 00:00:00+00:00
1,CSE2027002,2025-01-02 00:00:00+00:00
2,CSE2027003,2025-01-03 00:00:00+00:00
3,CSE2027004,2025-01-04 00:00:00+00:00
4,CSE2027005,2025-01-05 00:00:00+00:00


In [12]:
# STEP 12 — Select the final feature dataset columns
feature_columns = [
    "student_id",
    "Specialization",
    "CGPA",
    "Programming_Skill_Score",
    "Communication_Skill_Score",
    "Certifications_Count",
    "Internship_Flag",
    "Projects_Completed",
    "DSA_Platform_Rating",
    "Placement_Training_Flag",
    "Soft_Skills_Score",
    "Curriculum_Coverage_Percent",
    "Industry_Demand_Score",
    "Curriculum_Industry_Alignment_Score",
    "Experience_Score",
    "event_timestamp",
    "Skill_Gap_Label"
]

feature_df = feature_df[feature_columns].copy()

# Make the Feast numeric types explicit.
for col in [
    "Programming_Skill_Score", "Communication_Skill_Score",
    "Certifications_Count", "Internship_Flag", "Projects_Completed",
    "DSA_Platform_Rating", "Placement_Training_Flag", "Soft_Skills_Score",
    "Curriculum_Coverage_Percent", "Industry_Demand_Score"
]:
    feature_df[col] = feature_df[col].astype("int64")

feature_df["CGPA"] = feature_df["CGPA"].astype("float32")
feature_df["Curriculum_Industry_Alignment_Score"] = (
    feature_df["Curriculum_Industry_Alignment_Score"].astype("float32")
)

display(feature_df.head())
print("Final feature dataset shape:", feature_df.shape)

,student_id,Specialization,CGPA,Programming_Skill_Score,Communication_Skill_Score,Certifications_Count,Internship_Flag,Projects_Completed,DSA_Platform_Rating,Placement_Training_Flag,Soft_Skills_Score,Curriculum_Coverage_Percent,Industry_Demand_Score,Curriculum_Industry_Alignment_Score,Experience_Score,event_timestamp,Skill_Gap_Label
0,CSE2027001,Cloud Computing,8.25,33,65,1,1,2,1009,1,84,42,51,94.260002,76.666664,2025-01-01 00:00:00+00:00,Yes
1,CSE2027002,Core CSE,6.50,33,55,5,0,3,1719,0,50,84,77,94.849998,15.833333,2025-01-02 00:00:00+00:00,Yes
2,CSE2027003,Cloud Computing,6.17,73,43,0,0,1,1535,0,63,91,52,65.220001,2.500000,2025-01-03 00:00:00+00:00,Yes
3,CSE2027004,IoT,7.81,78,40,4,0,5,1982,1,38,42,92,65.440002,49.166668,2025-01-04 00:00:00+00:00,No
4,CSE2027005,IoT,6.74,40,59,6,1,6,1369,0,76,50,73,82.930000,65.000000,2025-01-05 00:00:00+00:00,Yes


Final feature dataset shape: (150, 17)


In [13]:
# STEP 13 — Create the Feast project folder
from pathlib import Path

repo = Path("student_skillgap_feast")
data_dir = repo / "data"
data_dir.mkdir(parents=True, exist_ok=True)

parquet_path = data_dir / "student_features.parquet"
feature_df.to_parquet(parquet_path, index=False)

print("Saved:", parquet_path)
print("Parquet rows:", len(pd.read_parquet(parquet_path)))

Saved: student_skillgap_feast/data/student_features.parquet
Parquet rows: 150


In [14]:
# STEP 14 — Create feature_store.yaml
feature_store_yaml = '''
project: student_skill_gap
registry: data/registry.db
provider: local

online_store:
    type: sqlite
    path: data/online_store.db
'''

(repo / "feature_store.yaml").write_text(feature_store_yaml.strip() + "\n")

print((repo / "feature_store.yaml").read_text())

project: student_skill_gap
registry: data/registry.db
provider: local

online_store:
    type: sqlite
    path: data/online_store.db



In [15]:
# STEP 15 — Create Feast entity, data source, and FeatureView
feature_definitions = '''
from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String


# Entity: one record belongs to one student.
student = Entity(
    name="student",
    join_keys=["student_id"],
    description="Student identifier"
)


# Data source: our Parquet feature data.
student_source = FileSource(
    name="student_skill_source",
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
)


# FeatureView: features that describe a student.
student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="Specialization", dtype=String),
        Field(name="CGPA", dtype=Float32),
        Field(name="Programming_Skill_Score", dtype=Int64),
        Field(name="Communication_Skill_Score", dtype=Int64),
        Field(name="Certifications_Count", dtype=Int64),
        Field(name="Internship_Flag", dtype=Int64),
        Field(name="Projects_Completed", dtype=Int64),
        Field(name="DSA_Platform_Rating", dtype=Int64),
        Field(name="Placement_Training_Flag", dtype=Int64),
        Field(name="Soft_Skills_Score", dtype=Int64),
        Field(name="Curriculum_Coverage_Percent", dtype=Int64),
        Field(name="Industry_Demand_Score", dtype=Int64),
        Field(name="Curriculum_Industry_Alignment_Score", dtype=Float32),
        Field(name="Experience_Score", dtype=Float32),
    ],
    online=True,
    source=student_source,
)
'''

(repo / "feature_definitions.py").write_text(feature_definitions.strip() + "\n")

print((repo / "feature_definitions.py").read_text())

from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String


# Entity: one record belongs to one student.
student = Entity(
    name="student",
    join_keys=["student_id"],
    description="Student identifier"
)


# Data source: our Parquet feature data.
student_source = FileSource(
    name="student_skill_source",
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
)


# FeatureView: features that describe a student.
student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="Specialization", dtype=String),
        Field(name="CGPA", dtype=Float32),
        Field(name="Programming_Skill_Score", dtype=Int64),
        Field(name="Communication_Skill_Score", dtype=Int64),
        Field(name="Certifications_Count", dtype=Int64),
        Field(name="Internship_Flag", dtype=Int64),
     

In [16]:
# STEP 16 — Register the Feast definitions
!cd student_skillgap_feast && feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [17]:
# STEP 17 — Verify the registered entity and FeatureView
!cd student_skillgap_feast && feast entities list
!cd student_skillgap_feast && feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [18]:
# STEP 18 — Create the Feast FeatureStore object
from feast import FeatureStore

store = FeatureStore(repo_path="student_skillgap_feast")

entity_df = feature_df[
    ["student_id", "event_timestamp", "Skill_Gap_Label"]
].copy()

display(entity_df.head())

,student_id,event_timestamp,Skill_Gap_Label
0,CSE2027001,2025-01-01 00:00:00+00:00,Yes
1,CSE2027002,2025-01-02 00:00:00+00:00,Yes
2,CSE2027003,2025-01-03 00:00:00+00:00,Yes
3,CSE2027004,2025-01-04 00:00:00+00:00,No
4,CSE2027005,2025-01-05 00:00:00+00:00,Yes


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
# STEP 19 — Retrieve historical features
feature_refs = [
    "student_skill_features:Specialization",
    "student_skill_features:CGPA",
    "student_skill_features:Programming_Skill_Score",
    "student_skill_features:Communication_Skill_Score",
    "student_skill_features:Certifications_Count",
    "student_skill_features:Internship_Flag",
    "student_skill_features:Projects_Completed",
    "student_skill_features:DSA_Platform_Rating",
    "student_skill_features:Placement_Training_Flag",
    "student_skill_features:Soft_Skills_Score",
    "student_skill_features:Curriculum_Coverage_Percent",
    "student_skill_features:Industry_Demand_Score",
    "student_skill_features:Curriculum_Industry_Alignment_Score",
    "student_skill_features:Experience_Score",
]

historical_df = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs
).to_df()

print("Historical feature dataset:")
display(historical_df.head(10))
print("Shape:", historical_df.shape)

Historical feature dataset:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,Skill_Gap_Label,Specialization,CGPA,Programming_Skill_Score,Communication_Skill_Score,Certifications_Count,Internship_Flag,Projects_Completed,DSA_Platform_Rating,Placement_Training_Flag,Soft_Skills_Score,Curriculum_Coverage_Percent,Industry_Demand_Score,Curriculum_Industry_Alignment_Score,Experience_Score
0,CSE2027001,2025-01-01 00:00:00+00:00,Yes,Cloud Computing,8.25,33,65,1,1,2,1009,1,84,42,51,94.260002,76.666664
1,CSE2027002,2025-01-02 00:00:00+00:00,Yes,Core CSE,6.50,33,55,5,0,3,1719,0,50,84,77,94.849998,15.833333
2,CSE2027003,2025-01-03 00:00:00+00:00,Yes,Cloud Computing,6.17,73,43,0,0,1,1535,0,63,91,52,65.220001,2.500000
3,CSE2027004,2025-01-04 00:00:00+00:00,No,IoT,7.81,78,40,4,0,5,1982,1,38,42,92,65.440002,49.166668
4,CSE2027005,2025-01-05 00:00:00+00:00,Yes,IoT,6.74,40,59,6,1,6,1369,0,76,50,73,82.930000,65.000000
5,CSE2027006,2025-01-06 00:00:00+00:00,Yes,Data Science,8.38,39,51,4,1,2,1746,0,58,83,70,87.820000,51.666668
6,CSE2027007,2025-01-07 00:00:00+00:00,Yes,AI & ML,8.84,59,34,6,0,6,1348,1,70,53,91,69.610001,55.000000
7,CSE2027008,2025-01-08 00:00:00+00:00,No,IoT,9.30,88,48,2,1,3,1949,0,84,97,87,92.400002,50.833332
8,CSE2027009,2025-01-09 00:00:00+00:00,Yes,IoT,6.44,47,95,3,1,0,1024,1,50,90,93,97.690002,75.000000
9,CSE2027010,2025-01-10 00:00:00+00:00,No,IoT,5.77,78,89,4,0,8,823,1,98,88,67,80.940002,56.666668


Shape: (150, 17)


In [20]:
# STEP 20 — Save historical features
historical_df.to_csv(
    "student_historical_features.csv",
    index=False
)

print("Historical feature dataset saved as student_historical_features.csv")

Historical feature dataset saved as student_historical_features.csv


In [21]:
# STEP 21 — Prepare X and y
model_features = [
    "Specialization",
    "CGPA",
    "Programming_Skill_Score",
    "Communication_Skill_Score",
    "Certifications_Count",
    "Internship_Flag",
    "Projects_Completed",
    "DSA_Platform_Rating",
    "Placement_Training_Flag",
    "Soft_Skills_Score",
    "Curriculum_Coverage_Percent",
    "Industry_Demand_Score",
    "Curriculum_Industry_Alignment_Score",
    "Experience_Score"
]

X = historical_df[model_features].copy()
y = historical_df["Skill_Gap_Label"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 120
Testing rows: 30


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
# STEP 22 — Preprocessing pipeline for the ML model
numeric_features = [
    "CGPA",
    "Programming_Skill_Score",
    "Communication_Skill_Score",
    "Certifications_Count",
    "Internship_Flag",
    "Projects_Completed",
    "DSA_Platform_Rating",
    "Placement_Training_Flag",
    "Soft_Skills_Score",
    "Curriculum_Coverage_Percent",
    "Industry_Demand_Score",
    "Curriculum_Industry_Alignment_Score",
    "Experience_Score"
]

categorical_features = ["Specialization"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ]
)

print("Preprocessing + model pipeline created.")

Preprocessing + model pipeline created.


In [23]:
# STEP 23 — Train the model
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
# STEP 24 — Evaluate the model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Model Accuracy (%):", round(accuracy * 100, 2))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Skill Gap", "Skill Gap"]
))

Model Accuracy: 0.9666666666666667
Model Accuracy (%): 96.67

Classification Report:
              precision    recall  f1-score   support

No Skill Gap       1.00      0.93      0.97        15
   Skill Gap       0.94      1.00      0.97        15

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
# STEP 25 — Materialize the features into the online store
!cd student_skillgap_feast && feast materialize 2025-01-01T00:00:00 2025-06-01T00:00:00

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [26]:
# STEP 26 — Check the Feast data directory
!ls -lh student_skillgap_feast/data

total 668K
-rw-r--r-- 1 root root 644K Aug 17 06:23 online_store.db
-rw-r--r-- 1 root root 2.0K Aug 17 06:23 registry.db
-rw-r--r-- 1 root root  18K Aug 17 06:21 student_features.parquet


In [27]:
# STEP 27 — Retrieve online features
test_student_id = feature_df["student_id"].iloc[0]

online_response = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {"student_id": test_student_id}
    ]
).to_dict()

print("Student:", test_student_id)
print("\nOnline feature output:")
for key, value in online_response.items():
    print(key, ":", value)

Student: CSE2027001

Online feature output:
student_id : ['CSE2027001']
Curriculum_Industry_Alignment_Score : [94.26000213623047]
Specialization : ['Cloud Computing']
Industry_Demand_Score : [51]
Programming_Skill_Score : [33]
Certifications_Count : [1]
Internship_Flag : [1]
Projects_Completed : [2]
Curriculum_Coverage_Percent : [42]
DSA_Platform_Rating : [1009]
CGPA : [8.25]
Experience_Score : [76.66666412353516]
Soft_Skills_Score : [84]
Communication_Skill_Score : [65]
Placement_Training_Flag : [1]


In [28]:
# STEP 28 — Convert online features to a DataFrame
online_df = pd.DataFrame(online_response)

print("Online feature DataFrame:")
display(online_df)

Online feature DataFrame:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,Curriculum_Industry_Alignment_Score,Specialization,Industry_Demand_Score,Programming_Skill_Score,Certifications_Count,Internship_Flag,Projects_Completed,Curriculum_Coverage_Percent,DSA_Platform_Rating,CGPA,Experience_Score,Soft_Skills_Score,Communication_Skill_Score,Placement_Training_Flag
0,CSE2027001,94.260002,Cloud Computing,51,33,1,1,2,42,1009,8.25,76.666664,84,65,1


In [29]:
# STEP 29 — Make one final prediction
online_model_df = online_df[model_features].copy()

final_prediction = model.predict(online_model_df)[0]
final_probability = model.predict_proba(online_model_df)[0][1]

label = "Yes" if final_prediction == 1 else "No"

print("Student ID:", test_student_id)
print("Predicted Skill Gap:", label)
print("Probability of Skill Gap:", round(final_probability, 4))

if label == "Yes":
    print("Final Prediction: The student is predicted to have a skill gap.")
else:
    print("Final Prediction: The student is predicted NOT to have a skill gap.")

Student ID: CSE2027001
Predicted Skill Gap: Yes
Probability of Skill Gap: 0.9945
Final Prediction: The student is predicted to have a skill gap.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
# STEP 30 — Save the trained model
import joblib

joblib.dump(model, "student_skill_gap_model.pkl")

print("Model saved as student_skill_gap_model.pkl")

Model saved as student_skill_gap_model.pkl


In [31]:
# STEP 31 — Show the final project structure
!find student_skillgap_feast -maxdepth 2 -type f | sort

student_skillgap_feast/data/online_store.db
student_skillgap_feast/data/registry.db
student_skillgap_feast/data/student_features.parquet
student_skillgap_feast/feature_definitions.py
student_skillgap_feast/feature_store.yaml
